In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*사용 예상 시간: Eagle 프로세서에서 8분 소요 (참고: 이는 예상치이며, 실제 실행 시간은 다를 수 있습니다.)*

## 배경

이 튜토리얼에서는 양자 회로에서 게이트를 절단하여 회로 깊이를 줄이는 `Qiskit 패턴`을 구축하는 방법을 설명합니다. 회로 절단에 대한 더 자세한 내용은 [circuit cutting Qiskit addon 문서](https://qiskit.github.io/qiskit-addon-cutting/)를 참고하세요.

## 요구 사항

이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요:
- Qiskit SDK v2.0 이상 ([시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함)
- Qiskit Runtime v0.22 이상 (`pip install qiskit-ibm-runtime`)
- Circuit cutting Qiskit addon v0.9.0 이상 (`pip install qiskit-addon-cutting`)

## 설정

In [1]:
import numpy as np

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import PauliList, Statevector, SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_addon_cutting import (
    cut_gates,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

## 1단계: 고전적 입력을 양자 문제로 매핑하기
Qiskit 패턴의 네 단계([문서](/guides/intro-to-patterns) 참조)를 사용하여 구현할 것입니다. 이 경우, 특정 깊이의 회로에서 기댓값을 시뮬레이션하기 위해 게이트를 절단하여 swap 게이트를 생성하고, 더 얕은 회로에서 하위 실험을 실행합니다. 게이트 절단은 2단계(비인접 게이트를 분해하여 양자 실행을 위한 회로 최적화)와 4단계(원래 회로의 기댓값 재구성을 위한 후처리)에 적용됩니다.
첫 번째 단계에서는 Qiskit 회로 라이브러리에서 회로를 생성하고 몇 가지 관측량을 정의합니다.

*   입력: 회로를 정의하는 고전적 매개변수
*   출력: 추상 회로 및 관측량

In [2]:
circuit = EfficientSU2(num_qubits=4, entanglement="circular").decompose()
circuit.assign_parameters([0.4] * len(circuit.parameters), inplace=True)
observables = PauliList(["ZZII", "IZZI", "IIZZ", "XIXI", "ZIZZ", "IXIX"])
circuit.draw("mpl", scale=0.8, style="iqp")

<Image src="../docs/images/tutorials/depth-reduction-with-circuit-cutting/extracted-outputs/54ed0f13-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/depth-reduction-with-circuit-cutting/extracted-outputs/54ed0f13-0.avif)

## 2단계: 양자 하드웨어 실행을 위한 문제 최적화
*   입력: 추상 회로 및 관측량
*   출력: 비인접 게이트를 절단하여 트랜스파일된 회로 깊이를 줄인 대상 회로 및 관측량

Qubit 3과 0 사이의 게이트를 실행하기 위해 두 번의 swap이 필요하고, Qubit을 초기 위치로 되돌리기 위해 두 번 더 swap이 필요한 초기 레이아웃을 선택합니다. 프리셋 패스 매니저에서 사용 가능한 가장 높은 수준의 최적화인 `optimization_level=3`을 선택합니다.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=circuit.num_qubits, simulator=False
)

pm = generate_preset_pass_manager(
    optimization_level=3, initial_layout=[0, 1, 2, 3], backend=backend
)
transpiled_qc = pm.run(circuit)

![swap이 필요한 Qubit을 보여주는 결합 맵](../docs/images/tutorials/depth-reduction-with-circuit-cutting/swaps.avif)

In [4]:
print(f"Transpiled circuit depth: {transpiled_qc.depth()}")
transpiled_qc.draw("mpl", scale=0.4, idle_wires=False, style="iqp", fold=-1)

Transpiled circuit depth: 103


<Image src="../docs/images/tutorials/depth-reduction-with-circuit-cutting/extracted-outputs/4fe4af43-1.avif" alt="Output of the previous code cell" />

*Find and cut the distant gates:* We will replace the distant gates (gates connecting non-local qubits, 0 and 3) with `TwoQubitQPDGate` objects by specifying their indices. `cut_gates` will replace the gates in the specified indices with `TwoQubitQPDGate` objects and also return a list of `QPDBasis` instances -- one for each gate decomposition. The `QPDBasis` object contains information about how to decompose the cut gates into single-qubit operations.

In [5]:
# Find the indices of the distant gates
cut_indices = [
    i
    for i, instruction in enumerate(circuit.data)
    if {circuit.find_bit(q)[0] for q in instruction.qubits} == {0, 3}
]

# Decompose distant CNOTs into TwoQubitQPDGate instances
qpd_circuit, bases = cut_gates(circuit, cut_indices)

qpd_circuit.draw("mpl", scale=0.8)

<Image src="../docs/images/tutorials/depth-reduction-with-circuit-cutting/extracted-outputs/23e3d25e-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/depth-reduction-with-circuit-cutting/extracted-outputs/4fe4af43-1.avif)

*비인접 게이트 찾기 및 절단:* 비인접 게이트(비로컬 Qubit 0과 3을 연결하는 게이트)를 인덱스를 지정하여 `TwoQubitQPDGate` 객체로 교체합니다. `cut_gates`는 지정된 인덱스의 게이트를 `TwoQubitQPDGate` 객체로 교체하고 각 게이트 분해에 대한 `QPDBasis` 인스턴스 목록을 반환합니다. `QPDBasis` 객체에는 절단된 게이트를 단일 Qubit 연산으로 분해하는 방법에 대한 정보가 포함되어 있습니다.

In [6]:
# Generate the subexperiments and sampling coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=qpd_circuit, observables=observables, num_samples=np.inf
)

![이전 코드 셀의 출력](../docs/images/tutorials/depth-reduction-with-circuit-cutting/extracted-outputs/23e3d25e-0.avif)

*Backend에서 실행할 하위 실험 생성*: `generate_cutting_experiments`는 `TwoQubitQPDGate` 인스턴스를 포함하는 회로와 `PauliList`로 표현된 관측량을 입력으로 받습니다.

전체 크기 회로의 기댓값을 시뮬레이션하기 위해, 분해된 게이트의 결합 준확률 분포에서 여러 하위 실험이 생성된 후 하나 이상의 Backend에서 실행됩니다. 분포에서 추출하는 샘플 수는 `num_samples`로 제어되며, 각 고유 샘플에 하나의 결합 계수가 주어집니다. 계수 계산 방법에 대한 자세한 내용은 [설명 자료](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html)를 참고하세요.

In [7]:
# Transpile the decomposed circuit to the same layout
transpiled_qpd_circuit = pm.run(subexperiments[100])

print(f"Original circuit depth after transpile: {transpiled_qc.depth()}")
print(
    f"QPD subexperiment depth after transpile: {transpiled_qpd_circuit.depth()}"
)
transpiled_qpd_circuit.draw(
    "mpl", scale=0.6, style="iqp", idle_wires=False, fold=-1
)

Original circuit depth after transpile: 103
QPD subexperiment depth after transpile: 46


<Image src="../docs/images/tutorials/depth-reduction-with-circuit-cutting/extracted-outputs/70e2f1b6-1.avif" alt="Output of the previous code cell" />

*비교를 위해, 비인접 게이트 절단 후 QPD 하위 실험이 더 얕아진 것을 확인합니다*: 다음은 QPD 회로에서 생성된 임의로 선택된 하위 실험의 예시입니다. 깊이가 절반 이상 감소했습니다. 더 깊은 회로의 기댓값을 재구성하기 위해 이러한 확률론적 하위 실험 중 많은 수를 생성하고 평가해야 합니다.

In [8]:
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")

Sampling overhead: 729.0


## Step 3: Execute using Qiskit primitives

Execute the target circuits ("subexperiments") with the Sampler Primitive.

*   Input: Target circuits
*   Output: Quasi-probability distributions

In [9]:
# Transpile the subexperiments to the backend's instruction set architecture (ISA)
isa_subexperiments = pm.run(subexperiments)

# Set up the Qiskit Runtime Sampler primitive.  For a fake backend, this will use a local simulator.
sampler = SamplerV2(backend)

# Submit the subexperiments
job = sampler.run(isa_subexperiments)

In [11]:
# Retrieve the results
results = job.result()

In [10]:
print(job.job_id())

czypg1r6rr3g008mgp6g


## 3단계: Qiskit 기본 요소를 사용하여 실행하기
Sampler Primitive를 사용하여 대상 회로("하위 실험")를 실행합니다.

*   입력: 대상 회로
*   출력: 준확률 분포

In [12]:
reconstructed_expvals = reconstruct_expectation_values(
    results,
    coefficients,
    observables,
)
# Reconstruct final expectation value
final_expval = np.dot(reconstructed_expvals, [1] * len(observables))
print("Final reconstructed expectation value")
print(final_expval)

Final reconstructed expectation value
1.0751342773437473


In [13]:
ideal_expvals = [
    Statevector(circuit).expectation_value(SparsePauliOp(observable))
    for observable in observables
]
print("Ideal expectation value")
print(np.dot(ideal_expvals, [1] * len(observables)).real)

Ideal expectation value
1.2283177520039992


## Tutorial survey

Please take this short survey to provide feedback on this tutorial. Your insights will help us improve our content offerings and user experience.

[Link to survey](https://your.feedback.ibm.com/jfe/form/SV_2ftYFf9t72yFNIO)